# Neural network classifier (scikit-learn)

A neural network (multi-layer perceptron) extends the linear classifier by stacking an affine transform, a non-linear activation (ReLU), and a second affine transform before the softmax, letting it learn non-linear decision boundaries — something a single linear classifier can't do. This notebook trains it using scikit-learn's `MLPClassifier` — see "Neural Network Classifier" for a from-scratch implementation.

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/04_neural-networks'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

In [ ]:
![ -f utils.py ] || gdown --id 1NUGtNIABwt4PSAkBXPcMtnKPPaZYmxP3 -O utils.py

## Artificial dataset

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from utils import standard_scaler

num_classes = 2
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)  # try increasing the noise to see what happens

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize X (try to comment this out and see what happens)
X_train, mean, std = standard_scaler(X_train)
X_test, _, _ = standard_scaler(X_test, mean, std)

print(f"Train set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# Plot the dataset
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolors="gray")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")

print(f"X_train_min: {X_train.min()}, X_train_max: {X_train.max()}")
print(f"ymin: {y.min()}, ymax: {y.max()}")

## Training

In [ ]:
from sklearn.neural_network import MLPClassifier

# Hiperparâmetros de treinamento
num_epochs = 1000
learning_rate = 5e-1  # try reducing this to 1e-2 to see how much slower training converges
hidden_size = 32

# Modelo: MLP com uma camada oculta (ReLU) + softmax na saída
model = MLPClassifier(
    hidden_layer_sizes=(hidden_size,),
    activation='relu',
    solver='sgd',
    alpha=0.0,                  # sem regularização L2, pra comparar com a versão from-scratch
    batch_size=len(X_train),    # gradiente descendente "full-batch" (não minibatch)
    learning_rate_init=learning_rate,
    max_iter=num_epochs,
    tol=1e-6,                   # evita que o solver pare antes de rodar as num_epochs épocas
    n_iter_no_change=num_epochs,
    random_state=42,
    verbose=True,
)

model.fit(X_train, y_train)

In [ ]:
plt.plot(model.loss_curve_, label="Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss History")
plt.show()

## Test

In [ ]:
# Predict the labels
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[class_id for class_id in range(num_classes)])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[class_id for class_id in range(num_classes)])
disp.plot()
plt.show()

# Compute classification report
class_report = classification_report(y_test, y_pred, target_names=[str(class_id) for class_id in range(num_classes)])

print("\nClassification Report:")
print(class_report)

In [ ]:
def plot_decision_boundary(X, y, predict_fn, ax=None):
    if ax is None:
        ax = plt.gca()

    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 200), np.linspace(x2_min, x2_max, 200))
    grid_preds = predict_fn(np.c_[xx1.ravel(), xx2.ravel()])
    grid_preds = grid_preds.reshape(xx1.shape)

    ax.contourf(xx1, xx2, grid_preds, alpha=0.2, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors="gray", cmap="coolwarm")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")


import numpy as np

plot_decision_boundary(X_test, y_test, model.predict)
plt.title("Decision boundary (test set)")
plt.show()

accuracy = np.sum(y_pred == y_test) / len(y_test)
print("Accuracy:", accuracy)